# Before / after evaluation — base BLIP-2 vs fine-tuned LoRA

Produces the per-attribute accuracy table for the defense slide. Assumes
`train.ipynb` has produced `./adapter` and `./data/test.jsonl` exists.


In [ ]:
import json, os
import torch
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import PeftModel
from attributes import TARGET_FIELDS, parse_model_json

BASE_MODEL = "Salesforce/blip2-opt-2.7b"
ADAPTER_DIR = "./adapter"
DATA_DIR = "./data"
PROMPT = ("Describe this garment as JSON with keys subcategory, fabric, "
          "formality, neckline, sleeveLength, pattern.")
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = Blip2Processor.from_pretrained(BASE_MODEL)


In [ ]:
def load(base_only):
    m = Blip2ForConditionalGeneration.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
    if not base_only:
        m = PeftModel.from_pretrained(m, ADAPTER_DIR)
    return m.to(device).eval()

def evaluate(m, n=200):
    rows = [json.loads(l) for l in open(os.path.join(DATA_DIR, "test.jsonl"), encoding="utf-8")][:n]
    acc = {f: 0 for f in TARGET_FIELDS}
    for r in rows:
        img = Image.open(r["image"]).convert("RGB")
        inp = processor(images=img, text=PROMPT, return_tensors="pt").to(device)
        out = m.generate(**inp, max_new_tokens=128)
        pred = parse_model_json(processor.batch_decode(out, skip_special_tokens=True)[0])
        gold = json.loads(r["target"])
        for f in TARGET_FIELDS:
            acc[f] += int(str(pred.get(f)) == str(gold.get(f)))
    return {f: acc[f] / max(len(rows), 1) for f in TARGET_FIELDS}


In [ ]:
base_acc = evaluate(load(base_only=True))
ft_acc   = evaluate(load(base_only=False))

print(f"{'attribute':<14}{'base':>8}{'fine-tuned':>12}{'delta':>8}")
for f in TARGET_FIELDS:
    d = ft_acc[f] - base_acc[f]
    print(f"{f:<14}{base_acc[f]:>8.2f}{ft_acc[f]:>12.2f}{d:>+8.2f}")
